In [0]:
#scd1
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("scd1").getOrCreate()

data = [["1","finn","alberto","42000"],
        ["2","max","alberto","45000"],
        ["3","ana","alberto","52000"],
        ["4","elsa","alberto","32000"]]
columns = ["id","name","department","salary"]
df_full = spark.createDataFrame(data,columns)
columns = ["id","name","department","salary"]
data =  [["3","ana","calgary","92000"],
        ["4","else","winnipeg","15000"],
        ["5","ofs","toronto","82000"],
        ["2","max","alberto","49000"]]
columns = ["id","name","department","salary"]
df_daily_update = spark.createDataFrame(data,columns)
df_full.show()
df_daily_update.show()

from pyspark.sql.functions import coalesce
result = df_full.join(df_daily_update, on="id", how="full_outer").\
    select(coalesce(df_full.id, df_daily_update.id).alias("id"),\
        coalesce(df_daily_update.name,df_full.name).alias("name"),\
            coalesce(df_daily_update.department, df_full.department).alias("department"),\
                coalesce(df_daily_update.salary, df_full.salary).alias("salary"))
result.show()


In [0]:
#scd2
from pyspark.sql.functions import *
df_full = df_full.withColumn("active_flag", lit("Y")).withColumn("From_date", to_date(current_date())).\
    withColumn("to_date", lit(None))
df_full.show()

df_daily = df_daily_update.withColumn("active_flag", lit("Y")).withColumn("From_date", to_date(current_date())).\
    withColumn("to_date", lit(None))
df_daily.show()

#Create Dataframe by using Updating the Active Flag If any changes in dataframes using Hash
from pyspark.sql.functions import hash, lit, to_date, current_date
ds_update = df_full.join(df_daily, (df_full.id == df_daily.id) & (df_full.active_flag =="Y"), "inner") \
    .filter(hash(df_full.name, df_full.department, df_full.salary) != 
            hash(df_daily.name, df_daily.department, df_daily.salary)) \
    .select(df_full.id,
            df_full.name,
            df_full.department,
            df_full.salary,
            lit("N").alias("active_flag"),
            df_full.From_date,
            current_date().alias("to_date"))
ds_update.show()



In [0]:
#Create a Data Frame with No changes data using the update_ds dataframe
no_change = df_full.join(ds_update, (df_full.id == ds_update.id) & (df_full.active_flag == "Y"), "left_anti")
no_change.show()


In [0]:

#Create data frame using no_change Dataframe and the dataframe consists of the new records that need to insert
from pyspark.sql.functions import col
insert_df = df_daily.join(no_change, on="id", how="left_anti") \
    .select("id", "name", "department", "salary") \
    .withColumn("active_flag", lit("Y")) \
    .withColumn("From_date", current_date()) \
    .withColumn("to_date", lit(None).cast("date"))
insert_df.orderBy("id").show()

In [0]:
#Finally We have to concatenate update_ds , Insert_ds, no_change dataframe by using the union function.
df_final = ds_update.unionByName(insert_df).unionByName(no_change)
df_final.show()

In [0]:
insert_df.printSchema()